<a href="https://colab.research.google.com/github/nicotucci/TP-Laboratorio-AnalisisFutbol/blob/main/TP_LABORATORIO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Tecnicatura Universitaria en Gestión y Análisis de Datos en Organizaciones — FCE UBA**
###Trabajo Práctico Grupal · 1er Cuatrimestre 2026

---


**Repositorio:** [Link a Github](https://github.com/nicotucci/TP-Laboratorio-AnalisisFutbol)

#INICIO
Antes de comenzar con el analisis puro, de antemano haremos una observacion para justificar la eleccion del mismo.

In [ ]:
import pandas as pd
archivo = "https://raw.githubusercontent.com/nicotucci/TP-Laboratorio-AnalisisFutbol/main/Datasets/gastos%20en%20transf%20por%20liga%20-%20Sheet1.csv"
df = pd.read_csv(archivo)
display(df.head(8))

**A partir de la observacion de esta tabla que muestra los movimientos numericos en fichajes de las principales ligas, notamos que la Premier League presenta el mayor desembolso economico del mercado de pases, lo que la convierte en un caso de estudio relevante para analizar la relacion entre inversion y rendimiento.**

## 0. Configuración del entorno e Importacion de Librerias
> El notebook **carga la base directamente desde el repositorio de GitHub del grupo** (no hace falta
> subir nada a mano)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import re

# Para mostrar mejor los gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
url_posiciones = 'https://raw.githubusercontent.com/nicotucci/TP-Laboratorio-AnalisisFutbol/96a19b6bbce85288569fd91373c350642ec96c57/Datasets/premier%20posiciones%2022-23%20-%20premier%20posiciones%2022-23.xlsx%20(1).csv'
posiciones = pd.read_csv(url_posiciones)

print("Posiciones:")
display(posiciones.head(5))
display(posiciones.tail(5))


In [ ]:
posiciones.dtypes

In [ ]:
url_transfers = 'https://raw.githubusercontent.com/nicotucci/TP-Laboratorio-AnalisisFutbol/375751256a1c519d570caec12bbfde865f0a4a39/Datasets/transferencias_2022_2022.csv'
transfers = pd.read_csv(url_transfers)
print("Transferencias:")
display(transfers.head(7))
display(transfers.tail(7))

In [ ]:
transfers.dtypes

In [ ]:
# Celda 3: Exploración inicial
print("Dimensiones posiciones:", posiciones.shape[0],"filas y",posiciones.shape[1],"columnas")
print("")
print("Nulos posiciones:\n", posiciones.isnull().sum())
print("\nEstadísticas posiciones:")
display(posiciones.describe())

**Notamos que hay 399 valores nulos en la tabla de transferencias que vienen de la columna fee_cleaned (tarifa limpia). esto se debe a que son casos en los que son "loan transfer" o casos donde llegan porque se les finalizo el contrato**

In [ ]:
print("\nDimensiones transfers:", transfers.shape[0],"filas y",transfers.shape[1],"columnas")
print("")
print("Nulos transfers:\n", transfers.isnull().sum())

# ❓ Pregunta principal
## ✅ **¿Existe una correlacion estadisticamente significativa entre el gasto neto en transferencias y la posicion final en la tabla de la Premier League?**

### Subpreguntas q se pueden agregar:

- ¿El gasto neto explica diferencias en posición final o hay outliers que desafían esa relación?

- ¿Cómo se compara la eficiencia de gasto entre clubes que gastaron mucho vs clubes con gasto neto negativo (vendieron más de lo que compraron)?

- ¿Qué clubes fueron más eficientes en transformar dinero en puntos?

### b)¿Porqué vale la pena responderla?

La relación entre inversión en fichajes y rendimiento deportivo es una cuestión central en la gestión de clubes de fútbol. Frecuentemente, se asume que un mayor gasto neto (compras menos ventas) conduce a mejores posiciones en la tabla. Sin embargo, esta creencia rara vez se contrasta estadísticamente. Responder a la pregunta de si existe una correlación significativa entre el gasto neto y la posición final en la Premier League permite:

- Evaluar si los recursos destinados a transferencias netas se traducen efectivamente en éxito deportivo.

- Orientar la toma de decisiones de directivos: ¿deberían priorizar el gasto neto o factores como la masa salarial o la estabilidad del entrenador?

- Aportar evidencia cuantitativa a un debate mediático y de aficionados

### c)¿Qué respuesta encontraron?


**Encontramos evidencia de que no existe una relación lineal significativa**

El gráfico y la correlación numérica sugieren que, si bien la inversión en fichajes puede ser una estrategia, no es el único ni el principal determinante del éxito en la Premier League. Hay clubes que logran mucho con poco y otros que gastan mucho con resultados decepcionantes.

#Limpieza de Dataset



###1) Manejo de NULOS y agregado de columna

en fee_cleaned los valores que hayan estado vacios asignarle 0. y crear la columna valor_real para pasar ese fee_cleaned a millones

In [ ]:
# Limpieza de transferencias

# NUEVA COLUMNA: valor_real
# Si fee_cleaned está vacío/NaN → 0, si no → *1.000.000
transfers['valor_real'] = transfers['fee_cleaned'].apply(
    lambda x: 0 if pd.isna(x) or x == '' else x * 1_000_000
)

# dejarlo como entero para no tener .0
transfers['valor_real'] = transfers['valor_real'].astype('int64')
transfers['fee_cleaned'] = transfers['fee_cleaned'].fillna(0)

print("Fees extraídos. Ejemplo:")
filtrado = transfers[transfers['club_name'] == 'AFC Bournemouth']
display(filtrado[['club_name', 'transfer_movement', 'fee', 'fee_cleaned', 'valor_real']])

###2) Uso de GroupBy

Tabla con gastos e ingresos, y el gasto neto (ingresos-gasto) de cada club:

1.   Hacemos **Agrupaciones** por nombre de club, y para donde el 'transfer_movement' sea "in" (que ingresa al club (comprado) y otro para donde sea "out" (que sale del club (vendido)), sumar la columna "valor_real")
2.   A partir de lo calculado que fue GASTO e INGRESO, generamos la columna GASTO_NETO, que se calcula haciendo gasto-ingreso.

In [ ]:
# Celda 5: Agrupar por club y movimiento
gastos = transfers[transfers['transfer_movement'] == 'in'].groupby('club_name')['valor_real'].sum().reset_index()
gastos.columns = ['club', 'gasto_total_in_M']
ingresos = transfers[transfers['transfer_movement'] == 'out'].groupby('club_name')['valor_real'].sum().reset_index()
ingresos.columns = ['club', 'ingreso_total_out_M']

# Convertir a millones de euros para consistencia
gastos['gasto_total_in_M'] = gastos['gasto_total_in_M'] / 1_000_000
ingresos['ingreso_total_out_M'] = ingresos['ingreso_total_out_M'] / 1_000_000

# Unir
finanzas = pd.merge(gastos, ingresos, on='club', how='outer').fillna(0)
finanzas['gasto_neto_M'] = finanzas['gasto_total_in_M'] - finanzas['ingreso_total_out_M']
# Redondear
finanzas = finanzas.round(2)
print("Resumen financiero por club:")
display(finanzas.sort_values('gasto_neto_M', ascending=False))

## Costes totales de transferencia entrantes frente a salientes por club


Grafico de barras agrupadas para mostrar el ingreso y gasto de cada club

In [ ]:
# Desintegra el DataFrame de finanzas para facilitar la representación gráfica con seaborn.
finanzas_melted = finanzas.melt(id_vars='club',
                                value_vars=['gasto_total_in_M', 'ingreso_total_out_M'],
                                var_name='Transfer_Type',
                                value_name='Amount_M_Euro')

# Cambiamos las etiquetas del DataFrame para que aparezcan correctamente en la leyenda automática
finanzas_melted['Transfer_Type'] = finanzas_melted['Transfer_Type'].map({
    'gasto_total_in_M': 'Gasto (In)',
    'ingreso_total_out_M': 'Ingreso (Out)'
})

plt.figure(figsize=(18, 9))

# Definimos los nuevos colores en la paleta usando los nuevos nombres
sns.barplot(
    x='club',
    y='Amount_M_Euro',
    hue='Transfer_Type',
    data=finanzas_melted,
    palette={'Gasto (In)': 'red', 'Ingreso (Out)': 'green'}
)

plt.xlabel('Club')
plt.ylabel('Monto (Millones €)')
plt.title('Gasto Total vs. Ingreso Total por Transferencias por Club - Premier League 22/23')
plt.xticks(rotation=90)

# Dejamos que plt.legend use las etiquetas reales del DataFrame para no romper los colores
plt.legend(title='Tipo de Transferencia')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Este gráfico de barras agrupadas muestra claramente el monto total que cada club gastó en nuevos jugadores y el monto total que recibió por la venta de jugadores en millones de euros

## 📊 Transferencias en la Premier League (de 2018 a 2023)
### Gráfico de gastos, ingresos y balance por temporada

In [ ]:
# %% [markdown]


#PUESTAS A MANO LOS DATOS PORQ ERAN POCOS

# Temporadas
temporadas = ['18/19', '19/20', '20/21', '21/22', '22/23']

# Datos en millones de €
gastos  = [1670, 1810, 1620, 1710, 3190]    # mil mill. → millones
ingresos = [577.52, 949.47, 453.96, 825.09, 1030]
balance = [-1095.29, -864.47, -1164.84, -884.16, -2151.51]

# %%
# Configuración del gráfico
x = np.arange(len(temporadas))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 7))

# Barras
bars1 = ax.bar(x - width, gastos, width, label='Gastos', color='#e74c3c', alpha=0.85)
bars2 = ax.bar(x, ingresos, width, label='Ingresos', color='#2ecc71', alpha=0.85)
bars3 = ax.bar(x + width, balance, width, label='Balance', color='#3498db', alpha=0.85)

# Línea de referencia en cero para el balance
ax.axhline(y=0, color='gray', linewidth=0.8, linestyle='--')

# Etiquetas y títulos
ax.set_xlabel('Temporada', fontsize=13, fontweight='bold')
ax.set_ylabel('Millones de €', fontsize=13, fontweight='bold')
ax.set_title('Movimiento de transferencias en la Premier League', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(temporadas, fontsize=12)
ax.legend(fontsize=12)

# Agregar valores sobre las barras
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        va = 'bottom' if height >= 0 else 'top'
        offset = 15 if height >= 0 else -15
        ax.annotate(f'{height:.0f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, offset if height >= 0 else -offset),
                    textcoords="offset points",
                    ha='center', va=va, fontsize=9, fontweight='bold')

add_labels(bars1)
add_labels(bars2)
add_labels(bars3)

# Ajustar límites del eje Y para dejar espacio a las etiquetas
y_min = min(balance) - 200
y_max = max(gastos) + 200
ax.set_ylim(y_min, y_max)

# Grid en Y
ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# %%
#print("📊 Resumen por temporada:")
#print(f"{'Temp.':<8} {'Gastos (M€)':<14} {'Ingresos (M€)':<16} {'Balance (M€)'}")
##for i, t in enumerate(temporadas):
  #  print(f"{t:<8} {gastos[i]:<14,.0f} {ingresos[i]:<16,.2f} {balance[i]:,.2f}")